In [1]:
from file_parser import parse, get_schema_from_copybook
from file_parser.parsers.polars import file_parser


In [2]:
from pathlib import Path


def repo_root() -> Path:
    """Project root (directory containing pyproject.toml), from any cwd."""
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "pyproject.toml").is_file():
            return path
    msg = "Could not find project root (no pyproject.toml in cwd or parents)"
    raise FileNotFoundError(msg)


DATA = repo_root() / "data"

INPUT_PATH = DATA / "huge_fixed_size_file.dat"
INTERMEDIATE_OUTPUT_PATH = DATA / "huge_fixed_size_file.parquet"
OUTPUT_PATH = DATA / "huge_fixed_size_file_validations.parquet"
FORMULAS_PATH = repo_root() / "formulas.txt"

In [3]:

COPYBOOK = """
       01  FILE-RECORD.
           05  FULL-NAME                  PIC X(50).
           05  YEAR                       PIC 9(4).
           05  AMOUNT                     PIC 9(09)V99.
"""

file_schema = get_schema_from_copybook(COPYBOOK)
input_df = parse(INPUT_PATH, INTERMEDIATE_OUTPUT_PATH, file_schema, file_parser)

input_df.head(10).collect()

FULL_NAME,YEAR,AMOUNT
str,i64,"decimal[11,2]"
"""RAHGTSYCLAFNAFROFPVAVSJEZJCCWQ…",6065,272956798.27
"""SSSBROGMHYSFIUBWVKBYPTFNXRDDUO…",1876,457025829.31
"""DLMZXHNEYXIRQEUOVOAIAZXWIBXZCN…",5539,773997527.93
"""OPATBBAINHNOTXPGMKCRJLXBRRBTVC…",2697,16276771.55
"""HVMLZ PTEI POUBPNXEZCFQSGDYGQQ…",5718,123732174.09
"""XJEWSQ RAWIRZDDCOHQTFRHNYWCLHA…",3441,329942252.06
"""SJPX UYGEVELEYVLSTGESKBMFYJWXG…",8379,520363024.25
"""YFBLOVSZTFLZYQRDYNSISOFRKUEPKW…",2414,43998735.51
"""GMWBLRFSNGRAUUCLEZNBGWMVS QZYQ…",3788,961344045.63


In [4]:
from formula_engine import compute

In [5]:
result_df = compute(FORMULAS_PATH, input_df)
result_df.head(10).collect()

FULL_NAME,YEAR,AMOUNT,VALID_NAME,VALID_YEAR,VALID_AMOUNT,IND_AMOUNT_PLUS_YEAR,IND_AMOUNT_PLUS_YEAR_DOUBLE
str,i64,"decimal[11,2]",bool,bool,bool,"decimal[38,2]",f64
"""RAHGTSYCLAFNAFROFPVAVSJEZJCCWQ…",6065,272956798.27,false,false,true,272962863.27,5.4593e8
"""SSSBROGMHYSFIUBWVKBYPTFNXRDDUO…",1876,457025829.31,false,false,true,457027705.31,9.1406e8
"""DLMZXHNEYXIRQEUOVOAIAZXWIBXZCN…",5539,773997527.93,false,false,false,774003066.93,1.5480e9
"""OPATBBAINHNOTXPGMKCRJLXBRRBTVC…",2697,16276771.55,false,false,true,16279468.55,3.2559e7
"""HVMLZ PTEI POUBPNXEZCFQSGDYGQQ…",5718,123732174.09,false,false,true,123737892.09,2.4748e8
"""XJEWSQ RAWIRZDDCOHQTFRHNYWCLHA…",3441,329942252.06,false,false,true,329945693.06,6.5989e8
"""SJPX UYGEVELEYVLSTGESKBMFYJWXG…",8379,520363024.25,false,false,true,520371403.25,1.0407e9
"""YFBLOVSZTFLZYQRDYNSISOFRKUEPKW…",2414,43998735.51,false,false,true,44001149.51,8.8002e7
"""GMWBLRFSNGRAUUCLEZNBGWMVS QZYQ…",3788,961344045.63,false,false,false,961347833.63,1.9227e9
